In [ ]:
!pip install -U bitsandbytes -q
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q accelerate scikit-learn pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 11.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.3 MB/s eta 0:00:00


In [ ]:
import sys
sys.modules["torchao"] = None   # avoid the torch.int1 crash from before

import os
import re
import gc
import json
import glob
import string
import shutil
import torch
import pandas as pd
from PIL import Image
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

In [ ]:
# ---------------------------------------------------------------------------
# 1. Choose which dataset to run this session. Only this dataset's zip is
#    extracted and only its closed.csv rows are used below.
# ---------------------------------------------------------------------------
DATASET = "Neck"   # <-- change to "BrainFace" for the second run (then RESTART RUNTIME)

assert DATASET in ("Neck", "BrainFace"), "DATASET must be 'Neck' or 'BrainFace'"

ZIP_FILES = {
    "BrainFace": "/content/BrainFace_Test_with_CoT.zip",
    "Neck": "/content/Neck_Test_with_CoT.zip",
}

import zipfile

zip_path = ZIP_FILES[DATASET]
extract_dir = f"/content/{DATASET}_Test_with_CoT"

if not os.path.exists(extract_dir):
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
else:
    print(f"Already extracted: {extract_dir}")

TEST_DATA_ROOT = extract_dir
print("\nTEST_DATA_ROOT =", TEST_DATA_ROOT)
print(f"\nContents of {TEST_DATA_ROOT}:")
print(" ", os.listdir(TEST_DATA_ROOT))

Extracting /content/Neck_Test_with_CoT.zip...

TEST_DATA_ROOT = /content/Neck_Test_with_CoT

Contents of /content/Neck_Test_with_CoT:
  ['Neck_Test_with_CoT']


In [ ]:
MAX_ROWS_PER_MODEL = None   # set e.g. 30 for a quick test run before the full set

In [ ]:
# ---------------------------------------------------------------------------
# 2. Load closed-question test data for the selected dataset only
#    (with the "CoT" column already filled in)
# ---------------------------------------------------------------------------
all_csvs = glob.glob(os.path.join(TEST_DATA_ROOT, "**", "closed.csv"), recursive=True)

print(f"Found {len(all_csvs)} closed.csv test files for {DATASET}:")
for p in all_csvs:
    print(" ", p)

frames = []
for csv_path in all_csvs:
    split_dir = os.path.dirname(csv_path)
    df = pd.read_csv(csv_path)
    df["split_dir"] = split_dir
    frames.append(df)

test_df = pd.concat(frames, ignore_index=True)
test_df = test_df[test_df["CoT"].notna() & (test_df["CoT"].str.strip() != "")]
if MAX_ROWS_PER_MODEL:
    test_df = test_df.head(MAX_ROWS_PER_MODEL)
print(f"\nTotal closed-ended test rows for {DATASET} (with valid CoT): {len(test_df)}")

IMG_COL = "image_file" if "image_file" in test_df.columns else "img_name"

def resolve_image_path(split_dir: str, img_name: str) -> str:
    flat_name = os.path.basename(str(img_name))
    for c in [os.path.join(split_dir, str(img_name)), os.path.join(split_dir, flat_name),
              os.path.join(split_dir, str(img_name).replace("/", "_"))]:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Could not find image for {img_name!r} in {split_dir}")

Found 1 closed.csv test files for Neck:
  /content/Neck_Test_with_CoT/Neck_Test_with_CoT/CT/test/closed.csv

Total closed-ended test rows for Neck (with valid CoT): 16


In [ ]:
# ---------------------------------------------------------------------------
# 3. Your exact system prompt
# ---------------------------------------------------------------------------
def systemPrompt(question, cot):
    return f"""
              Context:
                     You are a board-certified radiologist and Medical Visual Question Answering (MedVQA) expert with experience interpreting X-ray, CT, MRI, Ultrasound, and other medical images.

              Objective:
                      Answer the user's question by verifying whether it is supported by the visual evidence in the medical image.

              Inputs:
              Question: {question}

              You have to think step by step.

              Instructions:
              1. Examine the medical image carefully.
              2. Independently determine the relevant visual findings before considering the CoT.
              3. Compare your own observations with the provided CoT.
              4. If the CoT is inconsistent with the image, disregard it.
              5. Answer the question using the following evidence priority:
                1. Medical image (highest priority)
                2. User question
                3. CoT (only if verified by the image)
              6. Never fabricate findings or rely on assumptions.
              7. If the image does not provide sufficient evidence to support a "Yes" answer, return "No."

              Output Requirements:
              - Output exactly one word.
              - Do not provide explanations, punctuation, or additional text.

              Valid outputs:
              Yes
              No
             Step of thinking: {cot}
           """

In [ ]:
# ---------------------------------------------------------------------------
# 4. Answer normalization/matching (Yes/No specific)
# ---------------------------------------------------------------------------
def normalize_yesno(text: str) -> str:
    text = text.strip().lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    if text.startswith("yes"):
        return "yes"
    if text.startswith("no"):
        return "no"
    return text

In [ ]:
# ---------------------------------------------------------------------------
# 5. Model registry -- repo id + loader class + how to build/generate.
#    Each entry returns (model, processor) and a generate_fn(model,
#    processor, image, prompt_text) -> raw string output.
# ---------------------------------------------------------------------------
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

def _load_generic(repo, token=None):
    processor = AutoProcessor.from_pretrained(repo, token=token)
    model = AutoModelForImageTextToText.from_pretrained(
        repo, quantization_config=bnb_config, device_map="auto", token=token,
    )
    return model, processor

def load_llama_vision():
    # Unsloth's ungated public mirror of Llama-3.2-11B-Vision-Instruct --
    # avoids the 401 error from needing to accept Meta's license + authenticate.
    return _load_generic("unsloth/Llama-3.2-11B-Vision-Instruct")

def load_qwen25_vl():
    return _load_generic("Qwen/Qwen2.5-VL-7B-Instruct")

def load_qwen3_vl():
    return _load_generic("Qwen/Qwen3-VL-8B-Instruct")

def load_gemma4():
    # MODIFIED: was "google/gemma-4-31B-it" (dense 31B, ~18-20GB even in
    # 4-bit -- doesn't fit a free-tier T4's 16GB VRAM). Swapped for the
    # smaller multimodal Gemma 4 variant, which fits comfortably.
    return _load_generic("google/gemma-4-E4B-it")

def load_medgemma():
    # ADDED: Google's Gemma-3-based model fine-tuned specifically on medical
    # images (radiology, histopathology, dermatology, etc.) and medical VQA
    # data. Worth comparing against the general-purpose VLMs above on a
    # medical yes/no VQA task like this one -- and it's the smallest/cheapest
    # model here (4B), so it's low-cost to include.
    return _load_generic("google/medgemma-4b-it")

def generate_generic(model, processor, image, prompt_text):
    """Works for all models here -- they share the same chat-template +
    apply_chat_template + generate() pattern in recent transformers."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt_text},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)

    output_ids = model.generate(**inputs, max_new_tokens=8, do_sample=False)
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True).strip()

MODEL_REGISTRY = {
    "Gemma4-E4B":    load_gemma4,
    "MedGemma-4B":   load_medgemma,
    "Qwen2.5-VL-7B": load_qwen25_vl,
    "Llama-11B":     load_llama_vision,
    "Qwen3-VL-8B":   load_qwen3_vl,
}

In [ ]:
# ---------------------------------------------------------------------------
# 6. Run one model, one condition (with/without CoT), over the full test_df
# ---------------------------------------------------------------------------
def run_condition(model, processor, use_cot: bool):
    results = []
    for i, row in test_df.iterrows():
        try:
            img_path = resolve_image_path(row["split_dir"], row[IMG_COL])
            image = Image.open(img_path).convert("RGB")
            cot_text = row["CoT"] if use_cot else ""
            prompt_text = systemPrompt(row["question"], cot_text)
            raw_output = generate_generic(model, processor, image, prompt_text)
        except Exception as e:
            print(f"    [WARN] row {i} failed: {e}")
            raw_output = ""

        pred = normalize_yesno(raw_output)
        gold = normalize_yesno(str(row["answer"]))
        results.append({"question": row["question"], "gold": gold, "pred": pred, "raw": raw_output})

        if i % 20 == 0:
            print(f"    [{i}/{len(test_df)}] gold={gold} pred={pred} raw={raw_output!r}")
    return results

In [ ]:
# ---------------------------------------------------------------------------
# 7. Disk-cache cleanup between models (important on Colab free's limited disk)
# ---------------------------------------------------------------------------
def clear_model_cache(repo_id: str):
    """Delete this repo's downloaded weights from the local HF cache to
    free disk space before the next model loads."""
    cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
    folder_name = "models--" + repo_id.replace("/", "--")
    path = os.path.join(cache_dir, folder_name)
    if os.path.exists(path):
        size_gb = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, files in os.walk(path) for f in files
        ) / (1024**3)
        shutil.rmtree(path, ignore_errors=True)
        print(f"Cleared cache for {repo_id} (freed ~{size_gb:.1f} GB)")
    else:
        print(f"No cache found for {repo_id} (nothing to clear)")

In [ ]:
# ---------------------------------------------------------------------------
# 8. Run ALL models x BOTH conditions, collecting metrics for the summary
#    table -- for the DATASET selected in section 1 only.
# ---------------------------------------------------------------------------
summary_rows = []
all_results = {}

REPO_IDS = {
    "Gemma4-E4B": "google/gemma-4-E4B-it",
    "MedGemma-4B": "google/medgemma-4b-it",
    "Qwen2.5-VL-7B": "Qwen/Qwen2.5-VL-7B-Instruct",
    "Llama-11B": "unsloth/Llama-3.2-11B-Vision-Instruct",
    "Qwen3-VL-8B": "Qwen/Qwen3-VL-8B-Instruct",
}

for model_name, loader_fn in MODEL_REGISTRY.items():
    print(f"\n{'='*70}\nLoading {model_name}  [{DATASET}]\n{'='*70}")
    try:
        model, processor = loader_fn()
    except Exception as e:
        print(f"  [SKIPPING {model_name}] failed to load: {e}")
        clear_model_cache(REPO_IDS[model_name])
        continue

    try:
        for condition, use_cot in [("With", True), ("Without", False)]:
            print(f"\n--- {model_name} | CoT: {condition} | Dataset: {DATASET} ---")
            results = run_condition(model, processor, use_cot)
            all_results[f"{model_name}_{condition}"] = results

            y_true = [r["gold"] for r in results]
            y_pred = [r["pred"] for r in results]

            print(f"\nClassification report -- {model_name} ({condition} CoT, {DATASET}):")
            print(classification_report(y_true, y_pred, zero_division=0))
            acc = accuracy_score(y_true, y_pred) * 100

            labels_present = sorted(set(y_true) | set(y_pred))

            macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
                y_true, y_pred, labels=labels_present, average="macro", zero_division=0
            )
            weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
                y_true, y_pred, labels=labels_present, average="weighted", zero_division=0
            )
            per_class_p, per_class_r, per_class_f1, per_class_support = precision_recall_fscore_support(
                y_true, y_pred, labels=labels_present, zero_division=0
            )
            per_class_metrics = {
                label: {"precision": p, "recall": r, "f1": f1, "support": int(s)}
                for label, p, r, f1, s in zip(labels_present, per_class_p, per_class_r, per_class_f1, per_class_support)
            }

            print(f"Accuracy: {acc:.2f}%")

            summary_rows.append({
                "Dataset": DATASET,
                "Model Name": model_name,
                "CoT": condition,
                "Accuracy (%)": round(acc, 2),
                "Precision (macro)": round(macro_p * 100, 2),
                "Recall (macro)": round(macro_r * 100, 2),
                "F1 (macro)": round(macro_f1 * 100, 2),
                "Precision (weighted)": round(weighted_p * 100, 2),
                "Recall (weighted)": round(weighted_r * 100, 2),
                "F1 (weighted)": round(weighted_f1 * 100, 2),
                "Yes-class P/R/F1": (
                    f"{per_class_metrics.get('yes', {}).get('precision', 0)*100:.1f}/"
                    f"{per_class_metrics.get('yes', {}).get('recall', 0)*100:.1f}/"
                    f"{per_class_metrics.get('yes', {}).get('f1', 0)*100:.1f}"
                ),
                "No-class P/R/F1": (
                    f"{per_class_metrics.get('no', {}).get('precision', 0)*100:.1f}/"
                    f"{per_class_metrics.get('no', {}).get('recall', 0)*100:.1f}/"
                    f"{per_class_metrics.get('no', {}).get('f1', 0)*100:.1f}"
                ),
            })

            pd.DataFrame(summary_rows).to_csv(f"/content/cot_comparison_summary_{DATASET}.csv", index=False)
            with open(f"/content/cot_comparison_raw_results_{DATASET}.json", "w") as f:
                json.dump(all_results, f, indent=2)

    finally:
        del model, processor
        gc.collect()
        torch.cuda.empty_cache()
        clear_model_cache(REPO_IDS[model_name])


Loading Gemma4-E4B  [Neck]


processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

...


--- Qwen3-VL-8B | CoT: With | Dataset: Neck ---


/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


    [0/16] gold=no pred=no raw='No'

Classification report -- Qwen3-VL-8B (With CoT, Neck):
              precision    recall  f1-score   support

          no       0.82      1.00      0.90         9
         yes       1.00      0.71      0.83         7

    accuracy                           0.88        16
   macro avg       0.91      0.86      0.87        16
weighted avg       0.90      0.88      0.87        16

Accuracy: 87.50%

--- Qwen3-VL-8B | CoT: Without | Dataset: Neck ---


/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


    [0/16] gold=no pred=no raw='No'

Classification report -- Qwen3-VL-8B (Without CoT, Neck):
              precision    recall  f1-score   support

          no       0.70      0.78      0.74         9
         yes       0.67      0.57      0.62         7

    accuracy                           0.69        16
   macro avg       0.68      0.67      0.68        16
weighted avg       0.69      0.69      0.68        16

Accuracy: 68.75%
Cleared cache for Qwen/Qwen3-VL-8B-Instruct (freed ~32.7 GB)


In [ ]:
# ---------------------------------------------------------------------------
# 9. Final summary table for this DATASET + save results
# ---------------------------------------------------------------------------
summary_df = pd.DataFrame(summary_rows)
print("\n" + "=" * 100)
print(f"FULL SUMMARY TABLE -- {DATASET} -- all models, both conditions, all metrics")
print("=" * 100)
print(summary_df.to_string(index=False))

summary_df.to_csv(f"/content/cot_comparison_summary_{DATASET}.csv", index=False)
with open(f"/content/cot_comparison_raw_results_{DATASET}.json", "w") as f:
    json.dump(all_results, f, indent=2)

print(f"\nSaved full metrics table: /content/cot_comparison_summary_{DATASET}.csv")
print(f"Saved raw per-question predictions: /content/cot_comparison_raw_results_{DATASET}.json")
print("\nColumns in the summary table:")
print("  Accuracy (%)            -- overall exact-match accuracy")
print("  Precision/Recall/F1 (macro)    -- unweighted average across Yes/No classes")
print("  Precision/Recall/F1 (weighted) -- averaged, weighted by class support (sample count)")
print("  Yes-class P/R/F1        -- precision/recall/F1 specifically for 'Yes' answers")
print("  No-class P/R/F1         -- precision/recall/F1 specifically for 'No' answers")
next_dataset = "BrainFace" if DATASET == "Neck" else "Neck"
print(f"\nNext: Runtime -> Restart session, set DATASET = {next_dataset!r} in section 1, and run again.")


FULL SUMMARY TABLE -- Neck -- all models, both conditions, all metrics
Dataset    Model Name     CoT  Accuracy (%)  Precision (macro)  Recall (macro)  F1 (macro)  Precision (weighted)  Recall (weighted)  F1 (weighted)  Yes-class P/R/F1   No-class P/R/F1
   Neck    Gemma4-E4B    With         68.75              68.75           69.05       68.63                 69.53              68.75          68.87    62.5/71.4/66.7    75.0/66.7/70.6
   Neck    Gemma4-E4B Without         68.75              68.75           69.05       68.63                 69.53              68.75          68.87    62.5/71.4/66.7    75.0/66.7/70.6
   Neck Qwen2.5-VL-7B    With         81.25              87.50           78.57       79.22                 85.94              81.25          80.03   100.0/57.1/72.7   75.0/100.0/85.7
   Neck Qwen2.5-VL-7B Without         56.25              54.17           53.17       51.52                 54.69              56.25          53.41    50.0/28.6/36.4    58.3/77.8/66.7
   Neck     L